In [7]:
# import necessary part of dorieh python package 
from dorieh.cms.fts2yaml import MedicareFTS, mcr_type
import os
import pandas as pd
import re

ROOT_DATA_DIR = "/n/dominici_nsaph_l3/Lab/data/ci3_d_medicare/original_data/cms_medicare/data/"
# codes associated with each year
year_dict = {"2011": "4334",
            "2012": "4334",
            "2013": "4580",
            "2014": "5819",
            "2015": "7087",
            "2016": "8183",
            "2017": "10411",
            "2018": ""}

# years in which data is available for each file type
data_avail_dict = {"mbsf_ab_summary_res": {"min_year": 2011,
                                           "max_year": 2014},
                   "mbsf_abcd_summary_res": {"min_year": 2015,
                                             "max_year": 2018},
                   "medpar_all_file_res": {"min_year": 2011,
                                           "max_year": 2018},
                   "mbsf_d_cmpnts_res": {"min_year": 2011,
                                         "max_year": 2014}}

# file type list
ftype_lst = ["mbsf_ab_summary_res",
             "mbsf_d_cmpnts_res",
             "medpar_all_file_res",
            "mbsf_abcd_summary_res"]

In [8]:
# Takes a dictionary produced by fts.to_dict() and turns it into a data frame
def dict2df(dct):
    
    # First step: isolate column dict
    _,info_dct = next(iter(dct.items()))
    col_dat = info_dct["columns"]

    # iterate through columns
    dict_lst = []
    for dict_elem in col_dat:
        nm,info_dct = next(iter(dict_elem.items())) # extract column name and metadata
        out_dict = {"col_name": nm, 
                    "long_name": info_dct["description"].get("long_name", None),
                    "type": info_dct["type"], 
                    "description": info_dct["description"]["text"],
                    "original_type": info_dct["description"]["original_type"],
                    "width": info_dct["description"]["width"]}
        dict_lst.append(out_dict)
        
    return(pd.DataFrame(dict_lst))



# Takes dict of dfs and column binds them together, checking for matching column names
def make_merged_df(df_dict):
    # Initialize a variable to hold the merged result
    merged_df = None

    # Iterate through the dictionary and merge the data frames
    for key, df in df_dict.items():
        if merged_df is None:
            # For the first data frame, just set it as the merged_df
            merged_df = df
            merged_df["idx"] = merged_df.index
        else:
            # For subsequent data frames, merge with the existing merged_df
            merged_df = pd.merge(merged_df, df[["col_name", "type"]], 
                                 on='col_name', how="outer", suffixes=('', f'_{key}'))

    # adding column matching flag
    # Get all columns matching the pattern "type_*"
    columns_to_check = [col for col in merged_df.columns if re.match(r"type_\d{4}", col)]

    # Compare the 'type' column with all the matching columns
    merged_df["cols_match"] = merged_df[columns_to_check].eq(merged_df["type"], axis=0).all(axis=1)
        
    return(merged_df.sort_values("idx"))

In [9]:
# iterate through years and ftype to extract metadata

df_dict = {ftype: {} for ftype in ftype_lst}
for year in range(2011, 2019):
    yr_str = str(year)
    yr_code = year_dict[yr_str]
    
    # dealing with small inconsistencies in file names
    if year == 2018:
        f_suffix = f"000017155_req011836_{yr_str}.fts"
    elif len(yr_code) == 5:
        f_suffix = f"000017155_req0{yr_code}_{yr_str}.fts"
    else:
        f_suffix = f"000017155_req00{yr_code}_{yr_str}.fts"
        
    for ftype in ftype_lst:
        # check to make sure this dataset exists for given year
        if (year >= data_avail_dict[ftype]["min_year"] and 
            year <= data_avail_dict[ftype]["max_year"]):
            fname = f"{ROOT_DATA_DIR}/{yr_code}/{yr_str}/{ftype}{f_suffix}"
            # extract medicare data type
            medicare_type = mcr_type(ftype + f_suffix)
            fts = MedicareFTS(medicare_type).init(fname)
            df_dict[ftype][yr_str] = dict2df(fts.to_dict())

In [10]:
# get total number of variables in every df
out_dict = {}
for ftype in ftype_lst:
    dfs = df_dict[ftype]
    out_dict[ftype] = {"type": [ftype]*len(df_dict[ftype].keys()),
                       "total_vars": [len(df_dict[ftype][year]) for year in list(df_dict[ftype].keys())],
                       "year": list(df_dict[ftype].keys())}
    for each_row in zip(*([i] + (j) for i, j in out_dict[ftype].items())):
        print(*each_row, " ")

type total_vars year  
mbsf_ab_summary_res 56 2011  
mbsf_ab_summary_res 56 2012  
mbsf_ab_summary_res 56 2013  
mbsf_ab_summary_res 56 2014  
type total_vars year  
mbsf_d_cmpnts_res 81 2011  
mbsf_d_cmpnts_res 81 2012  
mbsf_d_cmpnts_res 81 2013  
mbsf_d_cmpnts_res 81 2014  
type total_vars year  
medpar_all_file_res 370 2011  
medpar_all_file_res 370 2012  
medpar_all_file_res 375 2013  
medpar_all_file_res 386 2014  
medpar_all_file_res 398 2015  
medpar_all_file_res 401 2016  
medpar_all_file_res 409 2017  
medpar_all_file_res 420 2018  
type total_vars year  
mbsf_abcd_summary_res 190 2015  
mbsf_abcd_summary_res 190 2016  
mbsf_abcd_summary_res 190 2017  
mbsf_abcd_summary_res 190 2018  


In [11]:
# looking at name consistency across years
# We'll first examine medpar
mdpr_df = make_merged_df(df_dict["medpar_all_file_res"])
mdpr_nomatch = (mdpr_df[mdpr_df["cols_match"] != True])
mdpr_nomatch = mdpr_nomatch.drop(["long_name", "description", "original_type"], axis=1)
mdpr_nomatch["missing_col"] = mdpr_nomatch.isna().any(axis=1)

# Fraction of variables consistent:
print(("Fraction of variables consistent: " + "1 - " + 
       str(len(mdpr_nomatch)) + "/" + str(len(mdpr_df)) + " = " + 
       str(1 - round(len(mdpr_nomatch)/len(mdpr_df), 3))))

# Fraction due to new columns:
print("Fraction of unmatched columns due to new columns: " + str(round(1 - (mdpr_nomatch["missing_col"].sum())/len(mdpr_nomatch), 3)))

# Printing columns with changes in type
mdpr_nomatch[mdpr_nomatch["missing_col"] != True]

Fraction of variables consistent: 1 - 55/421 = 0.869
Fraction of unmatched columns due to new columns: 0.073


,col_name,type,width,idx,type_2012,type_2013,type_2014,type_2015,type_2016,type_2017,type_2018,cols_match,missing_col
143,DRG_CD,VARCHAR(3),3.0,55.0,VARCHAR(3),VARCHAR(3),VARCHAR(3),VARCHAR(3),VARCHAR(3),VARCHAR(3),VARCHAR(4),False,False
20,BNDLD_MODEL_DSCNT_PCT,"NUMERIC(4,2)",4.2,329.0,"NUMERIC(4,2)","NUMERIC(4,2)","NUMERIC(7,4)","NUMERIC(7,4)","NUMERIC(7,4)","NUMERIC(7,4)","NUMERIC(7,4)",False,False
414,VBP_ADJSTMT_PCT,"NUMERIC(13,1)",13.1,330.0,"NUMERIC(13,1)","NUMERIC(13,1)","NUMERIC(15,12)","NUMERIC(15,12)","NUMERIC(15,12)","NUMERIC(15,12)","NUMERIC(15,12)",False,False
168,HRR_ADJSTMT_PCT,"NUMERIC(6,4)",6.4,331.0,"NUMERIC(6,4)","NUMERIC(6,4)","NUMERIC(8,5)","NUMERIC(8,5)","NUMERIC(8,5)","NUMERIC(8,5)","NUMERIC(8,5)",False,False
